# Setup

Clone [YOLOv5](https://github.com/ultralytics/yolov5/tree/v7.0) ที่ tag **`v7.0`** ซึ่งเป็นรีลีสสุดท้ายที่เป็น
**GPL-3.0** (ตั้งแต่ 14 เม.ย. 2023 repo เปลี่ยนเป็น AGPL-3.0) แล้วติดตั้ง
[dependencies](https://github.com/ultralytics/yolov5/blob/v7.0/requirements.txt) พร้อมตรวจ software และ hardware

> GPL-3.0 ใช้เชิงพาณิชย์ได้ แต่เป็น copyleft — ถ้า *แจกจ่าย* ซอฟต์แวร์ที่รวมโค้ดนี้ ต้องเปิดซอร์สส่วนนั้นด้วย
> (ไม่มีข้อบังคับ network-use แบบ AGPL จึงทำ API/SaaS ได้) — ไม่ใช่คำแนะนำทางกฎหมาย

In [ ]:
import os, sys, subprocess

# (1) torch >= 2.6 เปลี่ยนค่า default ของ torch.load เป็น weights_only=True ทำให้โหลด .pt ของ YOLOv5 ไม่ได้
#     ตัวแปรนี้เป็นของ PyTorch เอง จึงถูกส่งต่อไปยัง subprocess ทุกตัว (detect.py / train.py / export.py) อัตโนมัติ
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

REPO = '/content/yolov5'

# กันปัญหาโฟลเดอร์ซ้อน (yolov5/yolov5/yolov5) จากการรันเซลล์นี้ซ้ำ
if os.path.isdir(REPO + '/yolov5'):
    raise SystemExit('พบโฟลเดอร์ซ้อนจากการรันซ้ำ → สร้างเซลล์ใหม่ รัน  !rm -rf /content/yolov5  '
                     'แล้วกลับมารันเซลล์นี้อีกครั้ง')

if not os.path.isdir(REPO):
    !git clone -b v7.0 --depth 1 https://github.com/ultralytics/yolov5 {REPO}
%cd {REPO}

# (2) ไม่ให้ pip ลดเวอร์ชัน torch/torchvision ที่มากับ Colab (จะทำให้ GPU ใช้ไม่ได้)
!sed -i '/^torch/ s/^/# /' /content/yolov5/requirements.txt
%pip install -qr /content/yolov5/requirements.txt

# (3) setuptools รุ่นใหม่ถอด pkg_resources ออก แต่ utils/general.py ของ v7.0 ต้องใช้
try:
    import pkg_resources
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'setuptools<81'])

# (4) แก้ 3 จุดที่โค้ดปี 2022 เข้ากับไลบรารีปัจจุบันไม่ได้ — แก้ที่ source ตรง ๆ (รันซ้ำได้ ไม่ซ้อนทับ)
#     - NumPy 2.x ถอด np.trapz ซึ่ง utils/metrics.py ใช้คำนวณ AP  → mAP จะพังท้ายการเทรน
#     - NumPy 2.x ถอด np.float
#     - Pillow >= 10 ถอด ImageFont.getsize() → พังตอนวาดกล่องที่ชื่อคลาสไม่ใช่ ASCII (เช่นภาษาไทย)
!sed -i 's/np\.trapz(/(np.trapezoid if hasattr(np, "trapezoid") else np.trapz)(/' /content/yolov5/utils/metrics.py
!sed -i 's/\.astype(np\.float)/.astype(float)/' /content/yolov5/models/common.py
!sed -i 's/self\.font\.getsize(\(.*\))/(lambda _b: (_b[2] - _b[0], _b[3] - _b[1]))(self.font.getbbox(\1))/' /content/yolov5/utils/plots.py

# (5) ดึง pretrained weights จาก release v7.0 โดยตรง (กันโค้ด fallback ไปดึงจาก release ล่าสุดที่เป็น AGPL)
!wget -q -nc -P /content/yolov5 https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt
!wget -q -nc -P /content/yolov5 https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5n.pt

import torch, utils
display = utils.notebook_init()  # checks (ล้าง output ด้านบน จึงตรวจผลต่อจากนี้)

# ---- พิสูจน์ว่าแก้ได้จริง: ลองโหลด .pt ใน subprocess แบบเดียวกับที่ detect.py ทำ ----
_chk = subprocess.run([sys.executable, '-c',
    'import numpy as np, torch;'
    'assert hasattr(np, "trapz") or hasattr(np, "trapezoid");'
    'torch.load("yolov5s.pt", map_location="cpu");'
    'print("PASS")'], cwd=REPO, capture_output=True, text=True)
print('torch.load ใน subprocess :', 'PASS ✅' if 'PASS' in _chk.stdout else 'FAIL ❌')
assert 'PASS' in _chk.stdout, _chk.stderr[-1500:]
!grep -n "trapezoid" /content/yolov5/utils/metrics.py

# ---- ยืนยันว่าเป็น GPL-3.0 จริง ----
# หมายเหตุ: ห้ามเช็คด้วยการค้นคำว่า "affero" ทั้งไฟล์ เพราะตัวบท GPL-3.0 มาตรา 13
# อ้างถึง GNU Affero General Public License อยู่แล้ว 3 ครั้งตามปกติ → ต้องดูชื่อบนหัวไฟล์แทน
_l = [l.strip() for l in open('LICENSE').read().splitlines() if l.strip()][:2]
print('LICENSE :', ' | '.join(_l))
assert _l[0].upper() == 'GNU GENERAL PUBLIC LICENSE' and '29 June 2007' in _l[1], '❌ ไม่ใช่ GPL-3.0'
print('✅ GPL-3.0 confirmed (AGPL-3.0 ใช้ชื่อ GNU AFFERO... และลงวันที่ 19 November 2007)')

# 1. Predict ( Test Setup Yolo v5)

YOLOv5 เรียกใช้ผ่านสคริปต์ `detect.py` และรับ argument เพิ่มได้ เช่น `--img 640`
ดูรายการ argument ทั้งหมดและรายละเอียดอื่นได้ที่ [YOLOv5 Docs](https://github.com/ultralytics/yolov5/tree/v7.0#quick-start-examples)

In [ ]:
%cd /content/yolov5
# Run inference on an image with YOLOv5s
!python detect.py --weights yolov5s.pt --img 640 --conf 0.25 --source 'https://ultralytics.com/images/zidane.jpg'

import glob, os
from IPython.display import Image, display
display(Image(filename=max(glob.glob('runs/detect/*/zidane.jpg'), key=os.path.getmtime), width=700))

# 2. Add Data Roboflow

export จาก Roboflow ให้เลือกฟอร์แมต **YOLO v5 PyTorch** แล้วอัปโหลดไฟล์ `.zip` ที่ได้

In [ ]:
import zipfile
from google.colab import files

%cd /content
uploaded = files.upload()
for filename in uploaded.keys():
  pass

zip_ref = zipfile.ZipFile(filename, 'r')
zip_ref.extractall("/content/")
zip_ref.close()

# ตรวจว่าได้ data.yaml และมีรูปครบ (YOLOv5 อ่าน ../train/images ของ Roboflow ได้ตรงจากตำแหน่งนี้)
!cat /content/data.yaml
!echo "train images:" $(ls /content/train/images | wc -l) " | val images:" $(ls /content/valid/images 2>/dev/null | wc -l)

# 3. Train Data Roboflow

เลือกขนาดโมเดลได้ที่ `--weights` : `yolov5n.pt` (เล็กสุด เหมาะกับอุปกรณ์เล็ก) `yolov5s.pt` `yolov5m.pt` `yolov5l.pt` `yolov5x.pt`

In [ ]:
# Train YOLOv5s
%cd /content/yolov5
!python train.py --img 640 --batch 16 --epochs 200 --data /content/data.yaml --weights yolov5s.pt --cache

#4. Test Model Training

In [ ]:
%cd /content/yolov5
import glob, os
BEST = max(glob.glob('/content/yolov5/runs/train/*/weights/best.pt'), key=os.path.getmtime)
print('best.pt :', BEST)

!python detect.py --weights {BEST} --img 640 --conf 0.25 --source /content/test/images

from IPython.display import Image, display
out = max(glob.glob('/content/yolov5/runs/detect/*'), key=os.path.getmtime)
for f in sorted(glob.glob(out + '/*.jpg'))[:5]:
    display(Image(filename=f, width=700))

# 5. Export to Model

Export โมเดล YOLOv5 เป็นฟอร์แมตที่รองรับด้านล่างด้วย argument `--include` เช่น `--include onnx`
ดูรายละเอียดที่ [YOLOv5 Export](https://github.com/ultralytics/yolov5/issues/251)

- 💡 ProTip: Export เป็น [ONNX](https://onnx.ai/) หรือ [OpenVINO](https://docs.openvino.ai/latest/index.html) เร็วขึ้นบน CPU ได้ถึง 3x
- 💡 ProTip: Export เป็น [TensorRT](https://developer.nvidia.com/tensorrt) เร็วขึ้นบน GPU ได้ถึง 5x
- 💡 ProTip: สำหรับอุปกรณ์ขนาดเล็ก ใช้ `--include tflite --int8` จะได้ไฟล์เล็กลงราว 4 เท่า

| Format                                                                     | `--include`     | Model                     |
|----------------------------------------------------------------------------|-----------------|---------------------------|
| [PyTorch](https://pytorch.org/)                                            | -               | `yolov5s.pt`              |
| [TorchScript](https://pytorch.org/docs/stable/jit.html)                    | `torchscript`   | `yolov5s.torchscript`     |
| [ONNX](https://onnx.ai/)                                                   | `onnx`          | `yolov5s.onnx`            |
| [OpenVINO](https://docs.openvino.ai/latest/index.html)                     | `openvino`      | `yolov5s_openvino_model/` |
| [TensorRT](https://developer.nvidia.com/tensorrt)                          | `engine`        | `yolov5s.engine`          |
| [CoreML](https://github.com/apple/coremltools)                             | `coreml`        | `yolov5s.mlmodel`         |
| [TensorFlow SavedModel](https://www.tensorflow.org/guide/saved_model)      | `saved_model`   | `yolov5s_saved_model/`    |
| [TensorFlow GraphDef](https://www.tensorflow.org/api_docs/python/tf/Graph) | `pb`            | `yolov5s.pb`              |
| [TensorFlow Lite](https://www.tensorflow.org/lite)                         | `tflite`        | `yolov5s.tflite`          |
| [TensorFlow Edge TPU](https://coral.ai/docs/edgetpu/models-intro/)         | `edgetpu`       | `yolov5s_edgetpu.tflite`  |
| [TensorFlow.js](https://www.tensorflow.org/js)                             | `tfjs`          | `yolov5s_web_model/`      |
| [PaddlePaddle](https://github.com/PaddlePaddle)                            | `paddle`        | `yolov5s_paddle_model/`   |

In [ ]:
%cd /content/yolov5
# export เป็น TensorFlow GraphDef (.pb)
# models/tf.py ของ v7.0 เขียนไว้สำหรับ Keras 2 จึงต้องใช้ tf-keras + TF_USE_LEGACY_KERAS=1 กับ TF รุ่นใหม่
%pip install -q tf-keras
!TF_USE_LEGACY_KERAS=1 python export.py --weights {BEST} --img 640 --include pb

# ตัวเลือกอื่น (เอา # ออกเพื่อใช้งาน)
# !python export.py --weights {BEST} --img 640 --include onnx --simplify                 # ONNX
# !TF_USE_LEGACY_KERAS=1 python export.py --weights {BEST} --img 640 --include tflite --half                          # TFLite FP16
# !TF_USE_LEGACY_KERAS=1 python export.py --weights {BEST} --img 640 --include tflite --int8 --data /content/data.yaml # TFLite INT8 (เล็กสุด)

import glob, os
print('\nไฟล์ที่ export ได้:')
for f in sorted(glob.glob(os.path.dirname(BEST) + '/*')):
    print(f'  {os.path.basename(f):<40}{os.path.getsize(f)/1e6:7.2f} MB')